# Functions


In [7]:
import pandas as pd
import numpy as np
from pathlib import Path

# DATA LOADING
current_dir = Path.cwd()
project_root = current_dir.parents[2]
path_data = project_root / 'DATA_PPMI/Results/MODEL_DATA/DATA_test.csv'
data = pd.read_csv(path_data,index_col=0)

# FUNCIONES PARA CALCULAR DELTA Y CLASIFICACIÓN HY3

def compute_delta(df_prev, df_next,sufix):

    # Alinear por index (inner join automático)
    df_prev, df_next = df_prev.align(df_next, join="inner", axis=0)
    
    # Seleccionar solo columnas numéricas
    numeric_cols = df_prev.select_dtypes(include='number').columns
    
    # Calcular delta
    delta = df_next[numeric_cols] - df_prev[numeric_cols]
    
    # Renombrar columnas
    delta.columns = [f"{col}_delta_{sufix}" for col in numeric_cols]
    
    return delta

def HY3_classification(nhy):
    if nhy == 0:
        return 0  # Healthy
    elif nhy  in [1,2]:
        return 1  # Early Stage
    else:
        return 2  # Advanced Stage
    
def HY4_classification(nhy):
    if nhy == 0:
        return 0  # Healthy
    elif nhy  == 1:
        return 1  # Early Stage
    elif nhy == 2:
        return 2  # Mid Stage
    else:
        return 3  # Advanced Stage

def HY43_classification(nhy):
    if nhy == 0:
        return 0  # Healthy
    elif nhy == 1:
        return 1  # Early Stage
    else:
        return 2  # Advanced Stage
    
def HY2_classification(nhy):
    if nhy == 0:
        return 0  # Healthy
    else:
        return 1  # Advanced Stage

def classification_mcid(x):
    if x <= -3:
        return 0  # Clinically meaningful improvement
    elif x >= 4:
        return 2   # Clinically meaningful worsening
    else:
        return 1   # Stable / no clinically meaningful change
    

# groupby para calcular estadísticas por paciente y visita

def compute_group_stats(df, group_level="PATNO", EVENT_ID:list=None):
    df = df.copy()
    df = df.loc[df["EVENT_ID"].isin(EVENT_ID),:]
    df.drop(columns=["EVENT_ID"], inplace=True)
    
    df_grouped = df.groupby(level=group_level)
    df_mean = df_grouped.mean().add_suffix("_mean")
    df_min  = df_grouped.min().add_suffix("_min")
    df_max  = df_grouped.max().add_suffix("_max")
    df_var  = df_grouped.var().add_suffix("_var")
    df_std  = df_grouped.std().add_suffix("_std")

    return pd.concat([df_mean, df_min, df_max, df_var, df_std], axis=1)
    



In [8]:
data['EVENT_ID'].unique()

array(['V04', 'V06', 'V08', 'V10'], dtype=object)

# V08 + Deltas

In [14]:
data_V08_delta = data.copy().set_index('PATNO')

V10_cols = ['ENRLLRRK2', 'ENRLGBA', 'COHORT_DEFINITION', 'SEX', 'RAWHITE', 'EDUCYRS', 'AGE_AT_VISIT', "NHY"]
updrs3_cols = [
    'NP3SPCH','NP3FACXP','NP3RIGN','NP3RIGRU','NP3RIGLU','NP3RIGRL','NP3RIGLL',
    'NP3FTAPR','NP3FTAPL','NP3HMOVR','NP3HMOVL','NP3PRSPR','NP3PRSPL','NP3TTAPR',
    'NP3TTAPL','NP3LGAGR','NP3LGAGL','NP3RISNG','NP3GAIT','NP3FRZGT','NP3PSTBL',
    'NP3POSTR','NP3BRADY','NP3PTRMR','NP3PTRML','NP3KTRMR','NP3KTRML','NP3RTARU',
    'NP3RTALU','NP3RTARL','NP3RTALL','NP3RTALJ','NP3RTCON'
]

data_last = data_V08_delta[V10_cols + ['EVENT_ID'] + updrs3_cols].copy()
data_last['UPDRS3_total'] = data_last[updrs3_cols].sum(axis=1)
data_last.drop(columns=updrs3_cols, inplace=True)

# Nueva visita objetivo: V10
data_V10 = data_last.loc[data_last["EVENT_ID"] == 'V10', :].copy()
# Última visita histórica: V08
data_V08 = data_last.loc[data_last["EVENT_ID"] == 'V08', :].copy()

data_V10 = data_V10.sort_index()
data_V08 = data_V08.sort_index()

data_V10['Delta_UPDRS3_V10_V08'] = data_V10['UPDRS3_total'] - data_V08['UPDRS3_total']

data_V10["STAGE_HY3"] = data_V10["NHY"].apply(HY3_classification)
data_V10["STAGE_HY2"] = data_V10["NHY"].apply(HY2_classification)
data_V10["STAGE_HY4"] = data_V10["NHY"].apply(HY4_classification)
data_V10["STAGE_HY43"] = data_V10["NHY"].apply(HY43_classification)
data_V10["STAGE_MCID"] = data_V10["Delta_UPDRS3_V10_V08"].apply(classification_mcid)

data_V10.drop(columns=["EVENT_ID", 'COHORT_DEFINITION', "NHY"], inplace=True)

data_V08_delta.drop(
    columns=['ENRLLRRK2', 'ENRLGBA', 'COHORT_DEFINITION', 'SEX', 'RAWHITE', 'EDUCYRS', 'AGE_AT_VISIT', "NHY"],
    inplace=True
)

# Generar los df para delta desplazados:
# antes: BL, V04, V06
# ahora: V04, V06, V08
V04_data = data_V08_delta.loc[data_V08_delta["EVENT_ID"] == 'V04', :].copy()
V06_data = data_V08_delta.loc[data_V08_delta["EVENT_ID"] == 'V06', :].copy()
V08_data = data_V08_delta.loc[data_V08_delta["EVENT_ID"] == 'V08', :].copy()

delta_V06_V04 = compute_delta(V04_data, V06_data, "V06_V04")
delta_V08_V06 = compute_delta(V06_data, V08_data, "V08_V06")
delta_V08_V04 = compute_delta(V04_data, V08_data, "V08_V04")

# Last visit hist: ahora V08
data_v08 = data_V08_delta.loc[data_V08_delta["EVENT_ID"] == 'V08', :].copy()
data_v08.columns = data_v08.columns + "_V08"
data_v08.drop(columns=["EVENT_ID_V08"], inplace=True)

# Unir los df de delta con el df de la última visita
data_V08_delta = data_V10.merge(data_v08, left_index=True, right_index=True, how='inner')
data_V08_delta = data_V08_delta.merge(delta_V06_V04, left_index=True, right_index=True, how='inner')
data_V08_delta = data_V08_delta.merge(delta_V08_V06, left_index=True, right_index=True, how='inner')
data_V08_delta = data_V08_delta.merge(delta_V08_V04, left_index=True, right_index=True, how='inner')

# Guardado
data_V08_delta.drop(
    columns=["STAGE_HY3", "STAGE_HY2", "STAGE_HY4", "STAGE_HY43", "STAGE_MCID", "Delta_UPDRS3_V10_V08", "UPDRS3_total"]
).to_csv(project_root / 'SCRIPTS/PYTHON/MODEL_ANALYSIS/DATA/HY_SCALE/FULL_FEATURES/X_V08+Deltas_TEST.csv')

data_V08_delta["STAGE_HY3"].to_csv(project_root / 'SCRIPTS/PYTHON/MODEL_ANALYSIS/DATA/HY_SCALE/FULL_FEATURES/y_HY3_TEST.csv')
data_V08_delta["STAGE_HY2"].to_csv(project_root / 'SCRIPTS/PYTHON/MODEL_ANALYSIS/DATA/HY_SCALE/FULL_FEATURES/y_HY2_TEST.csv')
data_V08_delta["STAGE_HY4"].to_csv(project_root / 'SCRIPTS/PYTHON/MODEL_ANALYSIS/DATA/HY_SCALE/FULL_FEATURES/y_HY4_TEST.csv')
data_V08_delta["STAGE_HY43"].to_csv(project_root / 'SCRIPTS/PYTHON/MODEL_ANALYSIS/DATA/HY_SCALE/FULL_FEATURES/y_HY43_TEST.csv')
data_V08_delta["STAGE_MCID"].to_csv(project_root / 'SCRIPTS/PYTHON/MODEL_ANALYSIS/DATA/UPDRS3/FULL_FEATURES/y_MCID_TEST.csv')
data_V08_delta['Delta_UPDRS3_V10_V08'].to_csv(project_root / 'SCRIPTS/PYTHON/MODEL_ANALYSIS/DATA/UPDRS3/FULL_FEATURES/y_delta_UPDRS3_V10_V08_TEST.csv')
data_V08_delta['UPDRS3_total'].to_csv(project_root / 'SCRIPTS/PYTHON/MODEL_ANALYSIS/DATA/UPDRS3/FULL_FEATURES/y_UPDRS3_total_TEST.csv')

data_V08_delta.head()



,ENRLLRRK2,ENRLGBA,SEX,RAWHITE,EDUCYRS,AGE_AT_VISIT,UPDRS3_total,Delta_UPDRS3_V10_V08,STAGE_HY3,STAGE_HY2,...,NP3PTRMR_delta_V08_V04,NP3PTRML_delta_V08_V04,NP3KTRMR_delta_V08_V04,NP3KTRML_delta_V08_V04,NP3RTARU_delta_V08_V04,NP3RTALU_delta_V08_V04,NP3RTARL_delta_V08_V04,NP3RTALL_delta_V08_V04,NP3RTALJ_delta_V08_V04,NP3RTCON_delta_V08_V04
PATNO,,,,,,,,,,,,,,,,,,,,,
3003,0,0,0.0,1.0,16.0,60.7,59.0,13.0,2,1,...,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
3018,0,0,0.0,1.0,16.0,64.7,47.0,9.0,1,1,...,0.0,-1.0,-2.0,-1.0,-1.0,0.0,1.0,0.0,0.0,0.0
3051,0,0,1.0,1.0,18.0,75.7,33.0,11.0,1,1,...,-1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3054,0,0,1.0,1.0,16.0,78.3,32.0,8.0,1,1,...,-1.0,0.0,-1.0,-1.0,-1.0,0.0,0.0,-1.0,0.0,-2.0
3056,0,0,1.0,1.0,16.0,59.6,36.0,5.0,1,1,...,1.0,1.0,0.0,-1.0,0.0,-1.0,1.0,0.0,0.0,1.0


In [15]:
data_V10['Delta_UPDRS3_V10_V08']

PATNO
3003      13.0
3018       9.0
3051      11.0
3054       8.0
3056       5.0
          ... 
120622     1.0
121619    19.0
128196     3.0
134456     2.0
139860     0.0
Name: Delta_UPDRS3_V10_V08, Length: 508, dtype: float64

# V08 + Stats(V04+V06)

In [11]:
data_V08_stats = data.copy().set_index('PATNO')

V10_cols = ['ENRLLRRK2', 'ENRLGBA', 'SEX', 'RAWHITE', 'EDUCYRS', 'AGE_AT_VISIT']
data_V10 = data_V08_stats[V10_cols + ['EVENT_ID']].copy()
data_V10 = data_V10.loc[data_V10["EVENT_ID"] == 'V10', :].copy()  # New Visit
data_V10.drop(columns=["EVENT_ID"], inplace=True)

data_V08_stats.drop(
    columns=['ENRLLRRK2', 'ENRLGBA', 'COHORT_DEFINITION', 'SEX', 'RAWHITE', 'EDUCYRS', 'AGE_AT_VISIT', "NHY"],
    inplace=True
)

# Generar los df para estadísticas
# antes: entre BL y V04
# ahora: entre V04 y V06
stats = compute_group_stats(data_V08_stats, group_level="PATNO", EVENT_ID=['V04', 'V06'])

# última visita histórica: antes V06, ahora V08
data_v08 = data_V08_stats.loc[data_V08_stats["EVENT_ID"] == 'V08', :].copy()
data_v08 = data_v08.add_suffix("_V08")
data_v08.drop(columns=["EVENT_ID_V08"], inplace=True)

data_V08_stats = data_V10.merge(data_v08, left_index=True, right_index=True, how='inner')
data_V08_stats = data_V08_stats.merge(stats, left_index=True, right_index=True, how='inner')

data_V08_stats.to_csv(
    project_root / 'SCRIPTS/PYTHON/MODEL_ANALYSIS/DATA/HY_SCALE/FULL_FEATURES/X_V08+STATS_TEST.csv'
)

data_V08_stats.head()

,ENRLLRRK2,ENRLGBA,SEX,RAWHITE,EDUCYRS,AGE_AT_VISIT,MCAALTTM_V08,MCACUBE_V08,MCACLCKC_V08,MCACLCKN_V08,...,NP3PTRMR_std,NP3PTRML_std,NP3KTRMR_std,NP3KTRML_std,NP3RTARU_std,NP3RTALU_std,NP3RTARL_std,NP3RTALL_std,NP3RTALJ_std,NP3RTCON_std
PATNO,,,,,,,,,,,,,,,,,,,,,
3003,0,0,0.0,1.0,16.0,60.7,0.0,1.0,1.0,1.0,...,0.707107,0.707107,0.000000,0.707107,0.000000,0.000000,0.0,0.000000,0.0,0.000000
3018,0,0,0.0,1.0,16.0,64.7,0.0,1.0,1.0,1.0,...,0.707107,1.414214,0.000000,0.707107,0.000000,0.707107,0.0,0.000000,0.0,0.707107
3051,0,0,1.0,1.0,18.0,75.7,0.0,1.0,1.0,1.0,...,0.707107,0.000000,0.000000,0.000000,0.707107,0.000000,0.0,0.000000,0.0,0.000000
3054,0,0,1.0,1.0,16.0,78.3,0.0,0.0,1.0,1.0,...,0.707107,0.000000,0.707107,0.707107,0.000000,0.000000,0.0,0.707107,0.0,1.414214
3056,0,0,1.0,1.0,16.0,59.6,1.0,1.0,1.0,1.0,...,0.707107,0.707107,0.000000,0.000000,0.000000,0.707107,0.0,0.707107,0.0,0.707107


# Stats(V04+V06+V08)

In [12]:
data_stats = data.copy().set_index('PATNO')

V10_cols = ['ENRLLRRK2', 'ENRLGBA', 'SEX', 'RAWHITE', 'EDUCYRS', 'AGE_AT_VISIT']
data_V10 = data_stats[V10_cols + ['EVENT_ID']].copy()
data_V10 = data_V10.loc[data_V10["EVENT_ID"] == 'V10', :].copy()  # New Visit
data_V10.drop(columns=["EVENT_ID"], inplace=True)

data_stats.drop(
    columns=['ENRLLRRK2', 'ENRLGBA', 'COHORT_DEFINITION', 'SEX', 'RAWHITE', 'EDUCYRS', 'AGE_AT_VISIT', "NHY"],
    inplace=True
)

# Generar los df para estadísticas
# antes: BL, V04, V06
# ahora: V04, V06, V08
stats = compute_group_stats(data_stats, group_level="PATNO", EVENT_ID=['V04', 'V06', 'V08'])

data_stats = data_V10.merge(stats, left_index=True, right_index=True, how='inner')
data_stats.to_csv(project_root / 'SCRIPTS/PYTHON/MODEL_ANALYSIS/DATA/HY_SCALE/FULL_FEATURES/X_STATS_TEST.csv')
data_stats.head()

,ENRLLRRK2,ENRLGBA,SEX,RAWHITE,EDUCYRS,AGE_AT_VISIT,MCAALTTM_mean,MCACUBE_mean,MCACLCKC_mean,MCACLCKN_mean,...,NP3PTRMR_std,NP3PTRML_std,NP3KTRMR_std,NP3KTRML_std,NP3RTARU_std,NP3RTALU_std,NP3RTARL_std,NP3RTALL_std,NP3RTALJ_std,NP3RTCON_std
PATNO,,,,,,,,,,,,,,,,,,,,,
3003,0,0,0.0,1.0,16.0,60.7,0.666667,1.000000,1.0,1.0,...,0.57735,0.57735,0.000000,0.57735,0.00000,0.00000,0.00000,0.00000,0.0,0.000000
3018,0,0,0.0,1.0,16.0,64.7,0.666667,1.000000,1.0,1.0,...,0.57735,1.00000,1.154701,0.57735,0.57735,0.57735,0.57735,0.00000,0.0,0.577350
3051,0,0,1.0,1.0,18.0,75.7,0.666667,1.000000,1.0,1.0,...,0.57735,0.00000,0.000000,0.00000,0.57735,0.00000,0.00000,0.00000,0.0,0.000000
3054,0,0,1.0,1.0,16.0,78.3,0.666667,0.666667,1.0,1.0,...,0.57735,0.00000,0.577350,0.57735,0.57735,0.00000,0.00000,0.57735,0.0,1.154701
3056,0,0,1.0,1.0,16.0,59.6,1.000000,1.000000,1.0,1.0,...,0.57735,0.57735,0.000000,0.57735,0.00000,0.57735,0.57735,0.57735,0.0,0.577350
